# Descriptions by example

One `pd.Series` per cell, so the Czech a template produces can be read in practice. Values are
real SoD vocabulary; no API is called and nothing is written.

A description is **data**: each dataset renders its template once, at cleanup, into
`data/prompts/<variant>.csv`, and a run names that variant. This notebook renders the same
templates ad hoc, so you can see what any row would get.

| template | for | rendered into |
|---|---|---|
| `datasets/sod/self_description_v1.json` | a survey respondent | `data/prompts/self_description_v1.csv` |
| `datasets/grid/grid_description_v1.json` | a grid profile | a column of the grid CSV itself |

Sentences are **alternatives in priority order** — the first whose slots are all filled wins.
That is how a sentence degrades when an answer is missing and how the age priority (exact
`age` over `age_range`) is expressed. Filters carry the grammar: `years`, `in`, `have`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
sys.path.insert(0, str(ROOT))

from panel.describe.lang.cs import grammar as cz
from panel.describe.template import load_template, read_table
from datasets.sod.code.describe import FILTERS as SOD_FILTERS, usable as sod_usable

SURVEY = load_template(ROOT / "datasets/sod/self_description_v1.json")
GRID = load_template(ROOT / "datasets/grid/grid_description_v1.json")
GRID_FILTERS = {"years": cz.age_clause}


def show(template, filters, usable, label, row):
    """Print one case: which columns it carries, then the description it renders."""
    print(f"# {label}")
    print(f"  columns: {list(row.index)}")
    print(f"  -> {template.render(row, filters, usable)}")


show_survey = lambda label, row: show(SURVEY, SOD_FILTERS, sod_usable, label, row)
show_grid = lambda label, row: show(GRID, GRID_FILTERS, None, label, row)

SOD = {
    "gender": "muž",
    "age": 28,
    "education_level": "vysokoškolské vzdělání",
    "region": "Středočeský kraj",
    "district": "Nymburk",
    "town_size": "méně než 1.000 obyvatel",
    "employment_status": "zaměstnanec na plný úvazek",
    "income_range": "30.001 - 40.000 Kč",
    "living_standard": "spíše dobrou",
    "interest_in_politics": "velmi se zajímám",
}

GRID_SOD = {
    "gender": "žena",
    "education_level": "středoškolské vzdělání s maturitou",
    "age": 37,
    "income_range": "Do 15.000 Kč",
    "interest_in_politics": "spíše se zajímám",
}

## 1. The template itself

The whole prompt in one glance — which is the point of keeping it as data rather than as
control flow.

In [2]:
for block in SURVEY.sentences:
    for position, alternative in enumerate(block):
        print(("  " if position else "- ") + alternative)
print()
print("columns:", SURVEY.columns)
print("filters:", sorted(SURVEY.filters()))

- Jsem {gender}, {age|years}, mé vzdělání je {education_level}.
- Žiji {region|in}, v okresu {district} a obci o velikosti {town_size}.
- Z hlediska zaměstnání jsem {employment_status} a příjem naší domácnosti je {income_range}.
  Z hlediska zaměstnání jsem {employment_status}.
  Příjem naší domácnosti je {income_range}.
- {living_standard|have} životní úroveň.
- {interest_in_politics} o politiku.

columns: ('gender', 'age', 'education_level', 'region', 'district', 'town_size', 'employment_status', 'income_range', 'living_standard', 'interest_in_politics')
filters: ['have', 'in', 'years']


## 2. A survey respondent

The reference case: this is what a `self_description_v1` prompt looks like, minus the topic's
question.

In [3]:
show_survey("complete SoD profile", pd.Series(SOD))

# complete SoD profile
  columns: ['gender', 'age', 'education_level', 'region', 'district', 'town_size', 'employment_status', 'income_range', 'living_standard', 'interest_in_politics']
  -> Jsem muž, je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji ve Středočeském kraji, v okresu Nymburk a obci o velikosti méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 30.001 - 40.000 Kč. Mám spíše dobrou životní úroveň. Velmi se zajímám o politiku.


## 3. Missing and non-substantive answers

`income_range` is empty for 347 respondents and `employment_status` for 9; a `nevím` answer is
not usable either. The employment/income block has three alternatives, so it degrades to
whichever answer exists instead of dropping both.

In [4]:
show_survey("no income (347 real respondents)", pd.Series({**SOD, "income_range": float("nan")}))

# no income (347 real respondents)
  columns: ['gender', 'age', 'education_level', 'region', 'district', 'town_size', 'employment_status', 'income_range', 'living_standard', 'interest_in_politics']
  -> Jsem muž, je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji ve Středočeském kraji, v okresu Nymburk a obci o velikosti méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek. Mám spíše dobrou životní úroveň. Velmi se zajímám o politiku.


In [5]:
show_survey("no employment (9 real respondents)", pd.Series({**SOD, "employment_status": None}))

# no employment (9 real respondents)
  columns: ['gender', 'age', 'education_level', 'region', 'district', 'town_size', 'employment_status', 'income_range', 'living_standard', 'interest_in_politics']
  -> Jsem muž, je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji ve Středočeském kraji, v okresu Nymburk a obci o velikosti méně než 1.000 obyvatel. Příjem naší domácnosti je 30.001 - 40.000 Kč. Mám spíše dobrou životní úroveň. Velmi se zajímám o politiku.


In [6]:
show_survey("neither employment nor income", pd.Series({**SOD, "employment_status": None, "income_range": None}))

# neither employment nor income
  columns: ['gender', 'age', 'education_level', 'region', 'district', 'town_size', 'employment_status', 'income_range', 'living_standard', 'interest_in_politics']
  -> Jsem muž, je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji ve Středočeském kraji, v okresu Nymburk a obci o velikosti méně než 1.000 obyvatel. Mám spíše dobrou životní úroveň. Velmi se zajímám o politiku.


In [7]:
show_survey("living standard nevím", pd.Series({**SOD, "living_standard": "nevím"}))

# living standard nevím
  columns: ['gender', 'age', 'education_level', 'region', 'district', 'town_size', 'employment_status', 'income_range', 'living_standard', 'interest_in_politics']
  -> Jsem muž, je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji ve Středočeském kraji, v okresu Nymburk a obci o velikosti méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 30.001 - 40.000 Kč. Velmi se zajímám o politiku.


In [8]:
show_survey("interest nevím", pd.Series({**SOD, "interest_in_politics": "nevím"}))

# interest nevím
  columns: ['gender', 'age', 'education_level', 'region', 'district', 'town_size', 'employment_status', 'income_range', 'living_standard', 'interest_in_politics']
  -> Jsem muž, je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji ve Středočeském kraji, v okresu Nymburk a obci o velikosti méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 30.001 - 40.000 Kč. Mám spíše dobrou životní úroveň.


In [9]:
show_survey("middling living standard (the have filter negates)", pd.Series({**SOD, "living_standard": "ani dobrou, ani špatnou"}))

# middling living standard (the have filter negates)
  columns: ['gender', 'age', 'education_level', 'region', 'district', 'town_size', 'employment_status', 'income_range', 'living_standard', 'interest_in_politics']
  -> Jsem muž, je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji ve Středočeském kraji, v okresu Nymburk a obci o velikosti méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 30.001 - 40.000 Kč. Nemám ani dobrou, ani špatnou životní úroveň. Velmi se zajímám o politiku.


## 4. Region forms, and the preposition

`{region|in}` reads `datasets/sod/forms/regions.json`, which stores the whole prepositional
phrase — Czech vocalises `v` → `ve` before st- and zl-. An unknown region raises rather than
being mangled.

In [10]:
for region in ["Středočeský kraj", "Zlínský kraj", "Hlavní město Praha", "Kraj Vysočina"]:
    show_survey(region, pd.Series({**SOD, "region": region}))

# Středočeský kraj
  columns: ['gender', 'age', 'education_level', 'region', 'district', 'town_size', 'employment_status', 'income_range', 'living_standard', 'interest_in_politics']
  -> Jsem muž, je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji ve Středočeském kraji, v okresu Nymburk a obci o velikosti méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 30.001 - 40.000 Kč. Mám spíše dobrou životní úroveň. Velmi se zajímám o politiku.
# Zlínský kraj
  columns: ['gender', 'age', 'education_level', 'region', 'district', 'town_size', 'employment_status', 'income_range', 'living_standard', 'interest_in_politics']
  -> Jsem muž, je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji ve Zlínském kraji, v okresu Nymburk a obci o velikosti méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 30.001 - 40.000 Kč. Mám spíše dobrou životní úroveň. Velmi se zajímám o politiku.
# 

In [11]:
try:
    SURVEY.render(pd.Series({**SOD, "region": "Neexistující kraj"}), SOD_FILTERS, sod_usable)
except KeyError as error:
    print(f"KeyError: {error} - as expected, an unknown region is not guessed at")

KeyError: 'Neexistující kraj' - as expected, an unknown region is not guessed at


## 5. Age: exact, banded, and both

The survey's own band comes from `AGE2` (`"25–34 let"`); the grid's from its config
(`"30-44"`, `"60+"`). One filter parses all of them.

In [12]:
for value in [28, "18-29", "25–34 let", "60+", "Více než 60 let"]:
    print(f"  {value!r:18} -> {cz.age_clause(value)}")

  28                 -> je mi 28 let
  '18-29'            -> je mi mezi 18 a 29 lety
  '25–34 let'        -> je mi mezi 25 a 34 lety
  '60+'              -> je mi 60 a více let
  'Více než 60 let'  -> je mi více než 60 let


In [13]:
show_grid("grid profile, exact age", pd.Series(GRID_SOD))

# grid profile, exact age
  columns: ['gender', 'education_level', 'age', 'income_range', 'interest_in_politics']
  -> Jsem žena, je mi 37 let, mé vzdělání je středoškolské vzdělání s maturitou. Příjem naší domácnosti je Do 15.000 Kč. Spíše se zajímám o politiku.


In [14]:
show_grid("grid profile, banded age", pd.Series({**{k: v for k, v in GRID_SOD.items() if k != "age"}, "age_range": "30-44"}))

# grid profile, banded age
  columns: ['gender', 'education_level', 'income_range', 'interest_in_politics', 'age_range']
  -> Jsem žena, je mi mezi 30 a 44 lety, mé vzdělání je středoškolské vzdělání s maturitou. Příjem naší domácnosti je Do 15.000 Kč. Spíše se zajímám o politiku.


In [15]:
show_grid("both present - the first alternative wins, so exact age", pd.Series({**GRID_SOD, "age_range": "60+"}))

# both present - the first alternative wins, so exact age
  columns: ['gender', 'education_level', 'age', 'income_range', 'interest_in_politics', 'age_range']
  -> Jsem žena, je mi 37 let, mé vzdělání je středoškolské vzdělání s maturitou. Příjem naší domácnosti je Do 15.000 Kč. Spíše se zajímám o politiku.


In [16]:
show_grid("a config varying only two factors", pd.Series({"gender": "muž", "age_range": "45-59"}))

# a config varying only two factors
  columns: ['gender', 'age_range']
  -> 


## 6. Why each dataset names its own template

The survey template *would* render a grid row — the location and employment blocks simply drop
out. It is not used that way on purpose: a template per dataset keeps each one saying what its
own data means, and keeps a survey row that is missing answers from looking like a thin profile
(`_Refactor_Plan.md` §3.4).

In [17]:
show_survey("a grid row through the SURVEY template - shorter, but not wrong", pd.Series(GRID_SOD))

# a grid row through the SURVEY template - shorter, but not wrong
  columns: ['gender', 'education_level', 'age', 'income_range', 'interest_in_politics']
  -> Jsem žena, je mi 37 let, mé vzdělání je středoškolské vzdělání s maturitou. Příjem naší domácnosti je Do 15.000 Kč. Spíše se zajímám o politiku.


## 7. The stored table

What a run actually sends. `read_table` refuses a table that does not cover the rows about to
be run, so a missing description is an error before the money is spent rather than a `nan`
prompt in the middle of a batch.

In [18]:
path = ROOT / "data/clean_data.csv"
if path.exists():
    people = pd.read_csv(path, index_col=0)
    table = read_table("self_description_v1", people.index, ROOT / "data/prompts")
    print(f"{len(table)} descriptions, column {list(table.columns)}")
    for position in (0, 143, 2500):
        print(f"\n# clean_data row {position} (id {people.index[position]})")
        print(" ", table.iloc[position, 0])
else:
    print(f"{path} not present - run datasets/sod/prepare.ipynb first")

3880 descriptions, column ['self_description_v1']

# clean_data row 0 (id 60696)
  Jsem muž, je mi 28 let, mé vzdělání je vysokoškolské vzdělání. Žiji ve Středočeském kraji, v okresu Nymburk a obci o velikosti méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 30.001 - 40.000 Kč. Mám spíše dobrou životní úroveň. Velmi se zajímám o politiku.

# clean_data row 143 (id 64825)
  Jsem muž, je mi 43 let, mé vzdělání je základní + středoškolské vzdělání bez maturity. Žiji ve Středočeském kraji, v okresu Beroun a obci o velikosti méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 40.001 - 60.000 Kč. Nemám ani dobrou, ani špatnou životní úroveň. Spíše se nezajímám o politiku.

# clean_data row 2500 (id 68809)
  Jsem žena, je mi 33 let, mé vzdělání je základní + středoškolské vzdělání bez maturity. Žiji v Jihomoravském kraji, v okresu Vyškov a obci o velikosti 1.000 - 5.000 obyvatel. Z

In [19]:
if path.exists():
    try:
        read_table("self_description_v1", people.index[:10].append(pd.Index([-1])), ROOT / "data/prompts")
    except ValueError as error:
        print(f"ValueError: {error}")

ValueError: C:\Users\Martin\CoRe\Panel_LLM\data\prompts\self_description_v1.csv does not match the data: 1 rows without a description, 3870 descriptions without a row. Re-render the variant.
